In [1]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
import pickle

In [2]:
class Dataset_converter(Dataset):
    def __init__(self, data, token_size, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), token_size), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, token_size*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), token_size))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*token_size:(kk+1)*token_size] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [3]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, token_size=7):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, token_size)
        
    def forward(self, x, hw=None, short_term_memory=None):
        if hw is None:
            out, hw = self.rnn(x)
        else:   
            out, hw = self.rnn(x, hw)
            
        out = self.fc1(out[:,-1,:])
        return out, hw


In [4]:
def wake_period(model, train_data, token_size, working_memory, short_term_memory, lr=4e-4):
    data_set = Dataset_converter(train_data, token_size, working_memory, short_term_memory)
    train_loader = DataLoader(data_set, batch_size=1, shuffle=False)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95)
    criterion = torch.nn.CrossEntropyLoss()

    total = 0
    correct_train = np.zeros(1000,dtype=float)
    train_acc = []
    for (X_train, y_train) in train_loader:
        optimizer.zero_grad()
    
        if total == 0:
            predicted_y, hidden = model(X_train, short_term_memory=short_term_memory)
        else:
            predicted_y, hidden = model(X_train, hw=memory, short_term_memory=short_term_memory)

        
        loss = criterion(predicted_y, y_train)
        loss.backward(retain_graph=True)
        optimizer.step()

        with torch.no_grad():
            memory = hidden.clone()
            true_y = y_train.argmax(axis=1)
            estimated_y = predicted_y.argmax(axis=1)

            total += 1
            if true_y == estimated_y:
                correct_train[total%1000] = 1
            else:
                correct_train[total%1000] = 0


            train_acc.append(
                np.sum(correct_train)/total if total<1000 else np.sum(correct_train)/1000
            )
            
            # if total%1000 == 0:
            #     print(f'Iter : {total+1}, loss: {loss:.4f}, train accuracy: {train_acc[-1]:.4f}, test accuracy: {test_acc[-1]:.4f}')

    return model, train_acc

In [5]:
nodes = list(range(2,22,2))
layers = [1,2,3,4]
reps = 5
lr = 4e-4
result_train = {}
result_test = {}

total_samples = 50000
working_memory = 1
short_term_memory = 7
n_community = 2
n_members = 3

token_size = n_community*n_members+1
input_size = token_size*working_memory

for layer in layers:
    print("Doing layer ", layer)
    result_train[layer] = {}
    result_test[layer] = {}
    for node in nodes:
        print("Doing Nodes ", node)
        result_train[layer][node] = []
        result_test[layer][node] = []
        for _ in tqdm(range(reps)):
            train_data = get_sequence(total_samples, n_community, n_members)
            model = RNN(input_size, node, num_layers=layer, token_size=token_size)
            _, train_acc = wake_period(model, train_data, token_size, working_memory, short_term_memory, lr=lr)
            result_train[layer][node].append(train_acc)


with open('pickle_files/minimal_rnn_changing_nodes_layers.pickle', 'wb') as f:
    pickle.dump(result_train, f)

Doing layer  1
Doing Nodes  2


100%|██████████| 5/5 [01:25<00:00, 17.11s/it]


Doing Nodes  4


100%|██████████| 5/5 [01:19<00:00, 15.99s/it]


Doing Nodes  6


...

100%|██████████| 5/5 [03:35<00:00, 43.05s/it]


Doing Nodes  18


100%|██████████| 5/5 [03:43<00:00, 44.72s/it]


Doing Nodes  20


100%|██████████| 5/5 [03:43<00:00, 44.66s/it]
